In [2]:
%load_ext autoreload
%autoreload 2

from shapely.geometry import Point
import pystac_client
import planetary_computer as pc

from utils import (
    search_wide_area,
    choose_aoi,
    find_candidates,
    pick_scenes,
    load_scene,
)

IndentationError: expected an indented block after function definition on line 37 (utils.py, line 38)

## 1. Settings

All the choices of the analysis in one place. Both periods cover the same dry-season months (June to September), so seasonal effects don't get mixed up with the actual change.

In [1]:
REFERENCE_POINT = Point(-63.90, -8.76)   # known deforestation area, Rondônia
SEARCH_RADIUS_DEG = 0.5                  # wide area used only to choose the AOI
AOI_HALF_SIZE_DEG = 0.05                 # AOI of roughly 10 x 10 km

BASELINE_WINDOW = "2018-06-01/2018-09-30"
RECENT_WINDOW = "2023-06-01/2023-09-30"

# SCL classes treated as unusable: no data, saturated, shadow, clouds, cirrus, snow
BAD_SCL_CLASSES = [0, 1, 3, 8, 9, 10, 11]

BANDS = ["B02", "B03", "B04", "B08", "B8A", "B11", "B12", "SCL"]
# B08 (broad NIR, 10 m) is used for the spectral indices.
# B8A (narrow NIR, 20 m) is needed only by the foundation model section.

NameError: name 'Point' is not defined

In [9]:
# function that finds the scenes of interest 
def search_scene(reference_point,wide_search_area,recent_window,c,coll):
    candidates_nearby = c.search(
    collections=coll,
    intersects=mapping(wide_search_area),
    datetime=recent_window,
    query={"eo:cloud_cover": {"lt": 30}},
    ).items()
    candidates_nearby = list(candidates_nearby)
    return candidates_nearby


In [ ]:
candidates_near= search_scene(reference_p,wsa,recent_win, catalog,collect)

In [12]:
# pick the scene whose real footprint covers the most area, usually the one furthest from a tile edge
def footprint (candidates_n, half_s): 

    reference_scene = max(candidates_n, key=lambda item: shape(item.geometry).area)
    footprint = shape(reference_scene.geometry)
    inner_point = footprint.buffer(-0.15).representative_point()
    bbox = [inner_point.x - half_s, inner_point.y - half_s,
            inner_point.x + half_s, inner_point.y + half_s]
    aoi = box(*bbox)
    aoi_geojson = mapping(aoi)
    return bbox,aoi_geojson

bbox,aoi_geojson = footprint (candidates_near, half_size)

In [13]:
bbox

[-63.461313123204675,
 -9.592136744908483,
 -63.36131312320468,
 -9.492136744908482]

In [16]:
def find_candidates(time_window, cloud_ceiling=40, pool_size=30):
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        intersects=aoi_geojson,
        datetime=time_window,
        query={"eo:cloud_cover": {"lt": cloud_ceiling}},
    )
    found = list(search.items())
    found.sort(key=lambda item: item.properties["eo:cloud_cover"])
    return found[:pool_size]

baseline_candidates = find_candidates(baseline_win)
recent_candidates = find_candidates(recent_win)
print("baseline candidates:", len(baseline_candidates))
print("recent candidates:", len(recent_candidates))

baseline candidates: 18
recent candidates: 30


In [17]:
cloud_related_classes = [0, 1, 3, 8, 9, 10, 11]  # no data, saturated, shadow, clouds, cirrus, snow

def aoi_clear_fraction(item):
    scl = odc.stac.load([item], bands=["SCL"], bbox=bbox, resolution=20, chunks={})
    scl = scl.isel(time=0)["SCL"]
    return float((~scl.isin(cloud_related_classes)).mean().values)

def pick_scenes(candidates, how_many=5, min_clear=0.6):
    scored = [(aoi_clear_fraction(item), item) for item in candidates]

    by_datetime = {}
    for clear_fraction, item in scored:
        dt = item.properties["datetime"]
        if dt not in by_datetime or item.id > by_datetime[dt][1].id:
            by_datetime[dt] = (clear_fraction, item)
    unique_scenes = list(by_datetime.values())

    unique_scenes.sort(key=lambda x: x[0], reverse=True)
    good_ones = [item for clear_fraction, item in unique_scenes if clear_fraction >= min_clear]

    if len(good_ones) < how_many:
        print(f"only {len(good_ones)} scenes above {min_clear*100:.0f}% clear, using the best available instead")
        return [item for _, item in unique_scenes[:how_many]]
    return good_ones[:how_many]

print("baseline scenes:")
baseline_scenes = pick_scenes(baseline_candidates)
for item in baseline_scenes:
    print(" ", item.id, item.properties["datetime"][:10])

print("recent scenes:")
recent_scenes = pick_scenes(recent_candidates)
for item in recent_scenes:
    print(" ", item.id, item.properties["datetime"][:10])

total_scenes = len(baseline_scenes) + len(recent_scenes)
assert total_scenes >= 6, "not enough clean scenes, try widening the date range"
print(f"{total_scenes} acquisitions in total")

baseline scenes:
  S2A_MSIL2A_20180819T142751_R053_T20LMQ_20201011T035719 2018-08-19
  S2A_MSIL2A_20180829T142751_R053_T20LMQ_20201103T112224 2018-08-29
  S2A_MSIL2A_20180720T142751_R053_T20LMQ_20201011T175511 2018-07-20
  S2A_MSIL2A_20180908T142751_R053_T20LMQ_20201011T100319 2018-09-08
  S2A_MSIL2A_20180710T142751_R053_T20LMQ_20201011T140724 2018-07-10
recent scenes:
  S2B_MSIL2A_20230629T142719_R053_T20LMQ_20241002T184523 2023-06-29
  S2A_MSIL2A_20230724T142721_R053_T20LMQ_20230724T233545 2023-07-24
  S2A_MSIL2A_20230922T142711_R053_T20LMQ_20241026T093629 2023-09-22
  S2B_MSIL2A_20230828T142719_R053_T20LMQ_20241019T002816 2023-08-28
  S2B_MSIL2A_20230808T142719_R053_T20LMQ_20241030T021940 2023-08-08
10 acquisitions in total


In [18]:
bands_needed = ["B02", "B03", "B04", "B08", "B8A", "B11", "B12", "SCL"]
# B08 is the broad, 10m NIR band used for NDVI/EVI/BSI everywhere else in this notebook.
# B8A is the narrow, 20m NIR band. It's downloaded here too because it's the one the
# GFM bonus section actually needs, see the note in that section for why.

def download_scene(item):
    ds = odc.stac.load([item], bands=bands_needed, bbox=bbox, resolution=10, chunks={})
    return ds.isel(time=0)

all_scenes = [("baseline", item) for item in baseline_scenes] + [("recent", item) for item in recent_scenes]

scenes = {}
for period, item in all_scenes:
    scenes[item.id] = {
        "period": period,
        "date": item.properties["datetime"],
        "data": download_scene(item),
    }
    print(f"downloaded {item.id} ({period}, {item.properties['datetime'][:10]})")

item_by_id = {item.id: item for item in baseline_scenes + recent_scenes}

downloaded S2A_MSIL2A_20180819T142751_R053_T20LMQ_20201011T035719 (baseline, 2018-08-19)
downloaded S2A_MSIL2A_20180829T142751_R053_T20LMQ_20201103T112224 (baseline, 2018-08-29)
downloaded S2A_MSIL2A_20180720T142751_R053_T20LMQ_20201011T175511 (baseline, 2018-07-20)
downloaded S2A_MSIL2A_20180908T142751_R053_T20LMQ_20201011T100319 (baseline, 2018-09-08)
downloaded S2A_MSIL2A_20180710T142751_R053_T20LMQ_20201011T140724 (baseline, 2018-07-10)
downloaded S2B_MSIL2A_20230629T142719_R053_T20LMQ_20241002T184523 (recent, 2023-06-29)
downloaded S2A_MSIL2A_20230724T142721_R053_T20LMQ_20230724T233545 (recent, 2023-07-24)
downloaded S2A_MSIL2A_20230922T142711_R053_T20LMQ_20241026T093629 (recent, 2023-09-22)
downloaded S2B_MSIL2A_20230828T142719_R053_T20LMQ_20241019T002816 (recent, 2023-08-28)
downloaded S2B_MSIL2A_20230808T142719_R053_T20LMQ_20241030T021940 (recent, 2023-08-08)
